In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
# Define the LSTM model in PyTorch
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.relu(out[:, -1, :])  # Consider only the output of the last time step
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.softmax(out)
        return out


In [3]:
# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the model
input_size = 1  # MFCC feature size
hidden_size = 256
num_layers = 1
num_classes = 7
model = LSTMModel(input_size, hidden_size, num_layers, num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [4]:
def extract_mfcc(filename):
    y, sr = librosa.load(filename, duration=3, offset=0.5)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    return mfcc

In [5]:
# Function to predict emotion label from MFCC features using the trained model
def predict_emotion(model, mfcc_features):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(mfcc_features, dtype=torch.float32).unsqueeze(0).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        return predicted.item()

In [6]:
# Load your trained model
model = LSTMModel(input_size, hidden_size, num_layers, num_classes).to(device)
model.load_state_dict(torch.load('LSTM_model_pytorch.pth')) 
model.eval()


LSTMModel(
  (lstm): LSTM(1, 256, batch_first=True)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=7, bias=True)
  (relu): ReLU()
  (softmax): Softmax(dim=1)
)

In [7]:
# Path to the audio file you want to test
audio_file_path = r"C:\Users\Tharunraj_S\Desktop\Major project\codes\TESS Toronto emotional speech set data\OAF_Pleasant_surprise\OAF_calm_ps.wav"

In [8]:
# Extract MFCC features from the audio file
mfcc_features = extract_mfcc(audio_file_path)

In [9]:
mfcc_features = np.expand_dims(mfcc_features, -1)

In [23]:
mfcc_features.shape

(40, 1)

In [10]:
# Predict emotion label
predicted_label_index = predict_emotion(model, mfcc_features)

emotion_labels = ['angry', 'disguist', 'fear','happy', 'neutral', 'pleasant_surprised', 'sad']
predicted_emotion = emotion_labels[predicted_label_index]

print(f'Predicted emotion label: {predicted_emotion}')

Predicted emotion label: pleasant_surprised


In [41]:
import pyaudio
import numpy as np
import librosa

# Define function to preprocess audio data and make predictions
def process_audio_and_predict(model):
    # Set up PyAudio
    CHUNK = 1024
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 44100

    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    frames_per_buffer=CHUNK)

    print("* Recording")

    while True:
        data = stream.read(CHUNK)
        audio_data = np.frombuffer(data, dtype=np.int16)
        
        # Preprocess audio data (e.g., apply windowing, calculate MFCC features)
        mfcc_features = librosa.feature.mfcc(y=audio_data.astype(float), sr=RATE, n_mfcc=40)[:, 0]
        mfcc_features = np.expand_dims(mfcc_features, -1)
        
        # Make prediction using the model
        predicted_label_index = predict_emotion(model,mfcc_features)
        
        # Map predicted index to emotion label
        emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'pleasant_surprised', 'sad']
        predicted_emotion = emotion_labels[predicted_label_index]
        
        print("Predicted Emotion:", predicted_emotion)

    print("* Done recording")

    stream.stop_stream()
    stream.close()
    p.terminate()


In [42]:
process_audio_and_predict(model)

* Recording
Predicted Emotion: happy
Predicted Emotion: happy
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: fear
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: fear
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: happy
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: fear
Predicte

Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: fear
Predicted Emotion: fear
Predicted Emotion: neutral
Predicted Emotion: happy
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: disgust
Predicted Emotion: happy
Predicted Emotion: disgust
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: fear
Predicted Emotion: pleasant_surprised
Predicted Emotion: disgust
Predicted Emotion: fear
Predicted Emotion: happy
Predicted Emotion: pl

KeyboardInterrupt: 

In [43]:
import numpy as np
import librosa
import pyaudio
import time

# Define the sampling rate and duration for audio recording
RATE = 44100  # Sample rate
DURATION = 5  # Duration in seconds

# Initialize PyAudio
p = pyaudio.PyAudio()
stream = p.open(format=pyaudio.paInt16,
                channels=1,
                rate=RATE,
                input=True,
                frames_per_buffer=int(RATE/10))

print("Recording...")

# Initialize variables
audio_frames = []

start_time = time.time()

while time.time() - start_time < DURATION:
    # Read audio data from the stream
    data = stream.read(int(RATE/10))
    audio_frames.append(np.frombuffer(data, dtype=np.int16))

print("Recording stopped.")

# Close the stream and terminate PyAudio
stream.stop_stream()
stream.close()
p.terminate()

# Concatenate audio frames into a single numpy array
audio_data = np.concatenate(audio_frames)

# Preprocess audio data (e.g., apply windowing, calculate MFCC features)
mfcc_features = librosa.feature.mfcc(y=audio_data.astype(float), sr=RATE, n_mfcc=40)[:, 0]
mfcc_features = np.expand_dims(mfcc_features, -1)        
# Make prediction using the model
predicted_label_index = predict_emotion(model,mfcc_features)

# Map predicted index to emotion label
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'pleasant_surprised', 'sad']
predicted_emotion = emotion_labels[predicted_label_index]

print(f"Predicted emotion: {predicted_emotion}")


Recording...
Recording stopped.
Predicted emotion: happy
